In [1]:
import time
import numpy as np
import pandas as pd 
from tqdm import tqdm

from PI_API.vale_connect import ValeConnect

In [2]:
feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

feature_tag_mapping = {
    'Active Power': 'U-LGS1-Active-Power-AI',
    'Reactive Power': 'U-LGS1-Reactive-Power-AI',
    'Governor speed actual': 'U-LGS1-SI-81101-AI',
    'UGB X displacement': 'U-LGS1-UGB-X-PK-PK-70-AI',
    'UGB Y displacement': 'U-LGS1-UGB-Y-PK-PK-340-AI',
    'LGB X displacement': 'U-LGS1-GB-X-PK-PK-70-AI',
    'LGB Y displacement': 'U-LGS1-LGB-Y-PK-PK-340-AI',
    'TGB X displacement': 'U-LGS1-TGB-X-PK-PK-270-AI',
    'TGB Y displacement': 'U-LGS1-TGB-Y-PK-PK-340-AI',
    'Stator winding temperature 13': 'U-LGS1-TI-81104A-AI',
    'Stator winding temperature 14': 'U-LGS1-TI-81104B-AI',
    'Stator winding temperature 15': 'U-LGS1-TI-81104C-AI',
    'Surface Air Cooler Air Outlet Temperature': 'U-LGS1-TI-81104D-AI',
    'Surface Air Cooler Water Inlet Temperature': 'U-LGS1-TI-81104E-AI',
    'Surface Air Cooler Water Outlet Temperature': 'U-LGS1-TI-81104F-AI',
    'Stator core temperature': 'U-LGS1-TI-81104G-AI',
    'UGB metal temperature': 'U-LGS1-TI-81104H-AI',
    'LGB metal temperature 1': 'U-LGS1-TI-81104J-AI',
    'LGB metal temperature 2': 'U-LGS1-TI-81104K-AI',
    'LGB oil temperature': 'U-LGS1-TI-81104L-AI',
    'Penstock Flow': 'U-LGS1-FI-81101-AI',
    'Turbine flow': 'U-LGS1-FIT-431-AI',
    'UGB cooling water flow': 'U-LGS1-FIT-81103A-AI',
    'LGB cooling water flow': 'U-LGS1-FIT-81103B-AI',
    'Generator cooling water flow': 'U-LGS1-FIT-81103C-AI',
    'Governor Penstock Pressure': 'U-LGS1-PI-81101-AI',
    'Penstock pressure': 'U-LGS1-PT-81150-AI',
    'Opening Wicked Gate': 'U-LGS1-ZT-81101-AI',
    'UGB Oil Contaminant': 'U-LGS1-AY-81103B-AI',
    'Gen Thrust Bearing Oil Contaminant': 'U-LGS1-AY-81103C-AI'
}

tag_array = [feature_tag_mapping[feature] for feature in feature_set]

In [3]:
server_root = '142.40.33.208'
server_base = 'pti-pi'
conn = ValeConnect(server_root, server_base)

In [4]:
pi_tag = tag_array
time_list = ['2024-01-01 00:00:00','2024-11-30 23:59:59']

In [5]:
from datetime import datetime, timedelta

now = datetime.utcnow()
start_time = now - timedelta(hours=2)

time_list = [start_time.strftime('%Y-%m-%d %H:%M:%S'), now.strftime('%Y-%m-%d %H:%M:%S')]

print(time_list)

['2025-02-22 02:53:20', '2025-02-22 04:53:20']


In [6]:
measured_horizon = 60 * 2 * 1

In [31]:
master_pd = "";

for i in tqdm(range(len(pi_tag)), desc="Prcessing:"):
    x_tag_wid = conn.get_webid_point(pi_tag[i])

    # for limited data
    value_resp = conn.get_stream_rec_valuetimespan_pd(x_tag_wid, time_list)

    if i == 0:
        value_resp['Timestamps'] = pd.to_datetime(value_resp['Timestamps'])
        master_pd = value_resp
    else:
        master_pd = pd.concat([master_pd, value_resp['Values']], axis=1, join='inner')

master_pd = master_pd.values
master_pd = pd.DataFrame(data=master_pd, columns=['TimeStamp'] + feature_set)
df_sel = master_pd.iloc[-120:, :]
df_sel = df_sel.reset_index(drop=True)

Prcessing:: 100%|██████████| 30/30 [01:14<00:00,  2.49s/it]


In [ ]:
import time
from PI_API.vale_connect import ValeConnect

class TestValeAPI():

    def __init__(self):
        super(TestValeAPI, self).__init__()

        self.server_root = '142.40.33.208'

        self.server_base = 'pti-pi'
        self.x_tag = ['U-LGS1-GB-X-PK-PK-70-AI','U-LGS1-LGB-Y-PK-PK-340-AI',
                      'U-LGS1-TGB-X-PK-PK-270-AI','U-LGS1-TGB-Y-PK-PK-340-AI',
                      'U-LGS1-UGB-X-PK-PK-70-AI','U-LGS1-UGB-Y-PK-PK-340-AI']

        t = time.time()

        self.conn = ValeConnect(self.server_root,self.server_base)

        for i in self.x_tag:
            x_tag_wid = self.conn.get_webid_point(i)
            value_resp = self.conn.get_stream_rec_valuetime_pd(x_tag_wid)

            print(value_resp)
            print(type(value_resp['Timestamps'][1]))
            print(type(value_resp['Values'][1]))

        x_tag_wid = self.conn.get_webid_point(self.x_tag[0])
        time_list = ['2022-01-01 00:00:00','2022-01-01 01:00:00']
        result = self.conn.get_stream_rec_valuetimespan_pd(x_tag_wid,time_list)
        print(result)

        elapsed = time.time() - t
        print(elapsed)

if __name__ == "__main__":
    vale = TestValeAPI()